# Synthetic Transport Memory Generation

This notebook generates synthetic tracer transport data for validating the transport memory kernel framework.

Workflow:

1. Define time grid
2. Generate a true memory function H(t)
3. Generate a mobile transport kernel
4. Generate synthetic breakthrough curve (BTC)
5. Add measurement noise
6. Save synthetic data


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parents[0]
sys.path.append(str(ROOT))


## Import framework modules


In [ ]:
from src.memory_models.hybrid_memory import hybrid_model
from src.mobile_kernel.parametric_kernel import mobile_kernel
from src.mobile_kernel.kernel_matrix import build_convolution_matrix


## Generate true memory function

A hybrid memory model is used:

$$H(t)=A_1e^{-\beta t}+A_2(t+\epsilon)^{-\alpha}$$


In [ ]:
time = np.linspace(0, 100, 300)

H_true = hybrid_model(
    time,
    A1=0.6,
    beta=0.04,
    A2=0.4,
    alpha=0.5
)

plt.figure(figsize=(7,4))
plt.plot(time, H_true)
plt.xlabel('Time')
plt.ylabel('H(t)')
plt.title('True Memory Function')
plt.show()


## Generate mobile transport kernel and synthetic BTC


In [ ]:
g = mobile_kernel(
    time,
    m=2,
    b=0.05
)

K = build_convolution_matrix(g)

btc = K @ H_true

plt.figure(figsize=(7,4))
plt.plot(time, btc)
plt.xlabel('Time')
plt.ylabel('Concentration')
plt.title('Synthetic Breakthrough Curve')
plt.show()


## Save synthetic data


In [ ]:
output = Path('../data/synthetic/generated_btc')
output.mkdir(parents=True, exist_ok=True)

pd.DataFrame({
    'time': time,
    'btc': btc
}).to_csv(output/'synthetic_btc.csv', index=False)

print('Synthetic BTC saved successfully.')
